# 🧠 SPaRG-MF: Spiking Max-Former — CIFAR-100 Training

This notebook trains the **SPaRG-MF** (Spiking, Precision-Aware, Routed, and Gated Max-Former) on **CIFAR-100**.

### Setup
1. Go to `Runtime → Change runtime type → GPU` (T4 or better)
2. Run all cells in order
3. Checkpoints are saved to Google Drive so they survive disconnects

---

## 1. Mount Google Drive & Setup Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── EDIT THIS PATH to match where SPaRG-MF lives in YOUR Drive ───
PROJECT_DIR = '/content/drive/MyDrive/SPaRG-MF'
OUTPUT_DIR  = '/content/drive/MyDrive/SPaRG-MF/output'

import os
os.chdir(PROJECT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Working directory: {os.getcwd()}')
!ls -la

## 2. Install Dependencies

In [ ]:
!pip install -q timm>=0.9.0 spikingjelly>=0.0.0.0.14 einops

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 3. Verify Model Builds

Quick sanity check before committing to a long training run.

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from models.maxformer_snn import SpikingMaxFormer
from spikingjelly.activation_based import functional

test_model = SpikingMaxFormer(
    img_size=32, in_channels=3, num_classes=100,
    embed_dims=128, depths=[1, 1, 3], num_heads=4,
    time_steps=4, enable_head_gate=True,
    enable_token_gate=False, enable_mixed_prec=False,
).cuda()

total_params = sum(p.numel() for p in test_model.parameters())
print(f'✅ Model built! Parameters: {total_params / 1e6:.2f} M')

dummy = torch.randn(2, 3, 32, 32).cuda()
with torch.no_grad():
    out = test_model(dummy)
    functional.reset_net(test_model)
print(f'✅ Forward pass OK! {dummy.shape} → {out.shape}')

del test_model, dummy, out
torch.cuda.empty_cache()
print('✅ Ready to train!')

## 4. Training Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Edit these as needed
# ═══════════════════════════════════════════════════════════════

CONFIG = {
    # Model
    'embed_dims': 128,
    'depths': [1, 1, 3],
    'num_heads': 4,
    'time_steps': 4,
    'mlp_ratio': 4.0,

    # Training
    'batch_size': 64,           # T4: 64-128, A100: 256
    'epochs': 100,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'num_workers': 2,

    # Gating
    'enable_head_gate': True,
    'enable_token_gate': False,  # Too few tokens at 32x32
    'enable_mixed_prec': False,
}

print('Training Configuration:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## 5. 🚀 Train on CIFAR-100

In [ ]:
import sys, os, time
sys.path.insert(0, PROJECT_DIR)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from spikingjelly.activation_based import functional
from models.maxformer_snn import SpikingMaxFormer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

# ─── Data ───
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

train_set = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
test_set  = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)
train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'], shuffle=True,
                          drop_last=True, num_workers=CONFIG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_set, batch_size=CONFIG['batch_size'], shuffle=False,
                          num_workers=CONFIG['num_workers'], pin_memory=True)
print(f'CIFAR-100: {len(train_set)} train / {len(test_set)} test samples')

# ─── Model ───
model = SpikingMaxFormer(
    img_size=32, in_channels=3, num_classes=100,
    embed_dims=CONFIG['embed_dims'],
    depths=CONFIG['depths'],
    num_heads=CONFIG['num_heads'],
    time_steps=CONFIG['time_steps'],
    mlp_ratio=CONFIG['mlp_ratio'],
    enable_head_gate=CONFIG['enable_head_gate'],
    enable_token_gate=CONFIG['enable_token_gate'],
    enable_mixed_prec=CONFIG['enable_mixed_prec'],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model Parameters: {total_params / 1e6:.2f} M')

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

# ─── Resume from checkpoint if available ───
start_epoch = 1
best_acc = 0.0
ckpt_path = os.path.join(OUTPUT_DIR, 'cifar100_checkpoint.pth')
best_path = os.path.join(OUTPUT_DIR, 'cifar100_best.pth')

if os.path.isfile(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(ckpt['state_dict'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    best_acc = ckpt.get('best_acc', 0.0)
    print(f'✅ Resumed from epoch {ckpt["epoch"]} (best acc: {best_acc:.2f}%)')
    del ckpt
else:
    print('Starting training from scratch.')

# ─── Training Loop ───
print(f'\n{"="*60}')
print(f'Training SPaRG-MF on CIFAR-100 | Epochs {start_epoch}-{CONFIG["epochs"]}')
print(f'{"="*60}\n')

total_start = time.time()
for epoch in range(start_epoch, CONFIG['epochs'] + 1):
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    t0 = time.time()

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        functional.reset_net(model)

        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    scheduler.step()
    train_acc = 100. * correct / total

    # ─── Evaluation ───
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            functional.reset_net(model)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    test_acc = 100. * correct / total

    # ─── Logging ───
    is_best = test_acc > best_acc
    best_acc = max(best_acc, test_acc)
    marker = ' ★ NEW BEST' if is_best else ''
    print(f'Epoch {epoch:3d}/{CONFIG["epochs"]} | {time.time()-t0:.0f}s | '
          f'Train: {train_acc:.1f}% | Test: {test_acc:.1f}% | Best: {best_acc:.1f}%{marker}')

    # ─── Save checkpoint (every epoch, to Drive) ───
    state = {
        'epoch': epoch,
        'state_dict': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'acc': test_acc,
        'best_acc': best_acc,
    }
    torch.save(state, ckpt_path)
    if is_best:
        torch.save(state, best_path)

    # ─── Spike stats (every 10 epochs) ───
    if epoch % 10 == 0:
        stats = model.count_spikes()
        if stats:
            rates = [f"{k.split('.')[-1]}={v['avg_spike_rate']:.3f}" for k, v in list(stats.items())[:4]]
            print(f'  📊 Spike rates: {" | ".join(rates)}')

total_time = time.time() - total_start
print(f'\n{"="*60}')
print(f'🏆 Training Complete! Best Accuracy: {best_acc:.2f}% | Time: {total_time/3600:.1f}h')
print(f'   Checkpoint: {ckpt_path}')
print(f'   Best model: {best_path}')
print(f'{"="*60}')

## 6. Evaluate Best Model

In [ ]:
import sys, os
sys.path.insert(0, PROJECT_DIR)

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from spikingjelly.activation_based import functional
from models.maxformer_snn import SpikingMaxFormer

best_path = os.path.join(OUTPUT_DIR, 'cifar100_best.pth')
if not os.path.isfile(best_path):
    print(f'⚠️  No best model found at {best_path}')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = SpikingMaxFormer(
        img_size=32, in_channels=3, num_classes=100,
        embed_dims=CONFIG['embed_dims'],
        depths=CONFIG['depths'],
        num_heads=CONFIG['num_heads'],
        time_steps=CONFIG['time_steps'],
        mlp_ratio=CONFIG['mlp_ratio'],
        enable_head_gate=CONFIG['enable_head_gate'],
        enable_token_gate=CONFIG['enable_token_gate'],
        enable_mixed_prec=CONFIG['enable_mixed_prec'],
    ).to(device)

    ckpt = torch.load(best_path, map_location='cpu')
    model.load_state_dict(ckpt['state_dict'])
    print(f'Loaded best model from epoch {ckpt["epoch"]} (saved acc: {ckpt["acc"]:.2f}%)')
    model.eval()

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    ])
    test_set = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)
    test_loader = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            functional.reset_net(model)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    final_acc = 100. * correct / total
    print(f'\n🏆 Final Validation Accuracy: {final_acc:.2f}%')

    # Spike statistics
    stats = model.count_spikes()
    if stats:
        print('\n📊 Homeostatic Spike Rates:')
        for k, v in stats.items():
            print(f"  {k}: rate={v['avg_spike_rate']:.4f}, threshold={v['v_threshold']:.4f}")